# Polars Exercises 05 — Grouping & Aggregation

Grouping is how we answer business questions: *revenue per region*, *orders per customer*, *average basket per channel*. It replaces Excel's `SUMIF`, `COUNTIF` and PivotTables. The pattern is always `group_by(...).agg(...)`. We will also meet **window functions** (`.over(...)`), which aggregate *without collapsing* the rows.


**Data file:** `data/orders.csv`

---

### 1. Import Polars, read `data/orders.csv`, and add a `revenue` column (`quantity` × `unit_price`). Keep it in `orders` — we reuse it below.

In [1]:
import polars as pl

orders = pl.read_csv("data/orders.csv").with_columns(
    revenue=pl.col("quantity") * pl.col("unit_price")
)
orders.head()

order_id,customer_id,order_date,region,product_sku,category,product_name,quantity,unit_price,channel,status,discount_code,revenue
str,str,str,str,str,str,str,i64,f64,str,str,str,f64
"""ORD-000001""","""CUST-0171""","""2023-10-25""","""Asia Pacific""","""PB200""","""pens""","""Gel Pen 12-pack""",15,2.7,"""online""","""completed""","""SPRING10""",40.5
"""ORD-000002""","""CUST-0049""","""2023-12-02""","""North America""","""SH900""","""shirts""","""Cotton T-Shirt""",5,17.88,"""online""","""completed""","""BULK""",89.4
"""ORD-000003""","""CUST-0060""","""2023-02-27""","""Europe""","""BK300""","""books""","""Paperback Book""",16,7.89,"""online""","""completed""","""BULK""",126.24
"""ORD-000004""","""CUST-0065""","""2023-12-26""","""Latin America""","""SH900""","""shirts""","""Cotton T-Shirt""",37,17.06,"""online""","""completed""","""BULK""",631.22
"""ORD-000005""","""CUST-0036""","""2023-05-24""","""Asia Pacific""","""PT010""","""posters""","""Motivational Poster""",11,7.26,"""online""","""completed""",null,79.86


### 2. Total `quantity` **per category**.

In [2]:
orders.group_by("category").agg(pl.col("quantity").sum())

category,quantity
str,i64
"""shirts""",2782
"""notebooks""",3105
"""stickers""",1774
"""books""",3180
"""posters""",3517
"""pens""",3042
"""mugs""",1597


### 3. Number of orders **per region** (count the rows in each group).

In [3]:
orders.group_by("region").agg(pl.len().alias("n_orders"))

region,n_orders
str,u32
"""Middle East & Africa""",32
"""Asia Pacific""",224
"""Latin America""",110
"""North America""",393
"""Europe""",241


### 4. Average `unit_price` **per category**.

In [4]:
orders.group_by("category").agg(pl.col("unit_price").mean())

category,unit_price
str,f64
"""posters""",8.001444
"""mugs""",6.349091
"""pens""",1.906266
"""shirts""",19.452645
"""notebooks""",6.766802
"""stickers""",1.445957
"""books""",14.314146


### 5. Total `revenue` **per region**, sorted from highest to lowest.

In [5]:
orders.group_by("region").agg(pl.col("revenue").sum()).sort("revenue", descending=True)

region,revenue
str,f64
"""North America""",67476.21
"""Europe""",40689.42
"""Asia Pacific""",35481.23
"""Latin America""",18155.06
"""Middle East & Africa""",5494.14


### 6. **Per category**, compute three things at once: total quantity (`total_qty`), average price (`avg_price`), and order count (`n_orders`).

In [6]:
orders.group_by("category").agg(
    total_qty=pl.col("quantity").sum(),
    avg_price=pl.col("unit_price").mean(),
    n_orders=pl.len(),
)

category,total_qty,avg_price,n_orders
str,i64,f64,u32
"""pens""",3042,1.906266,158
"""stickers""",1774,1.445957,94
"""books""",3180,14.314146,164
"""notebooks""",3105,6.766802,172
"""posters""",3517,8.001444,180
"""shirts""",2782,19.452645,155
"""mugs""",1597,6.349091,77


### 7. Number of orders for **each combination** of `region` and `channel`.

In [7]:
orders.group_by("region", "channel").agg(pl.len().alias("n_orders"))

region,channel,n_orders
str,str,u32
"""Middle East & Africa""","""store""",11
"""Middle East & Africa""","""online""",19
"""North America""","""partner""",31
"""Latin America""","""partner""",9
"""North America""","""online""",224
…,…,…
"""North America""","""store""",138
"""Latin America""","""online""",71
"""Asia Pacific""","""partner""",21


### 8. Number of **distinct customers** per region (use `n_unique`).

In [8]:
orders.group_by("region").agg(pl.col("customer_id").n_unique().alias("n_customers"))

region,n_customers
str,u32
"""Middle East & Africa""",6
"""Latin America""",21
"""North America""",79
"""Asia Pacific""",42
"""Europe""",51


### 9. **Per category**, the minimum and maximum `quantity`.

In [9]:
orders.group_by("category").agg(
    min_qty=pl.col("quantity").min(),
    max_qty=pl.col("quantity").max(),
)

category,min_qty,max_qty
str,i64,i64
"""posters""",-5,39
"""shirts""",-5,38
"""stickers""",-3,39
"""pens""",-5,39
"""notebooks""",-5,39
"""mugs""",-5,39
"""books""",-5,39


### 10. Total `revenue` **per status**.

In [10]:
orders.group_by("status").agg(pl.col("revenue").sum())

status,revenue
str,f64
"""completed""",147611.89
"""cancelled""",9441.61
"""pending""",11607.05
"""returned""",-1364.49


### 11. **Per category**, the total revenue **of completed orders only** (aggregate with a filter inside: `pl.col("revenue").filter(...).sum()`).

In [11]:
orders.group_by("category").agg(
    completed_revenue=pl.col("revenue")
    .filter(pl.col("status") == "completed").sum()
)

category,completed_revenue
str,f64
"""posters""",26037.39
"""books""",40342.79
"""mugs""",7758.97
"""pens""",5200.99
"""stickers""",2045.12
"""shirts""",48904.08
"""notebooks""",17322.55


### 12. Total `revenue` **per month**. Derive the month from `order_date` and sort by month.

In [12]:
orders.group_by(
    month=pl.col("order_date").str.to_date("%Y-%m-%d").dt.month()
).agg(pl.col("revenue").sum()).sort("month")

month,revenue
i8,f64
1,15686.61
2,14923.21
3,11101.0
4,17577.25
5,11652.39
…,…
8,14558.22
9,11854.8
10,13719.5


### 13. The **top 3 categories** by total revenue.

In [13]:
orders.group_by("category").agg(pl.col("revenue").sum()).sort("revenue", descending=True).head(3)

category,revenue
str,f64
"""shirts""",54219.54
"""books""",45408.32
"""posters""",28442.0


### 14. The **average order value** (mean `revenue`) per `channel`.

In [14]:
orders.group_by("channel").agg(pl.col("revenue").mean().round(2).alias("avg_order_value"))

channel,avg_order_value
str,f64
"""partner""",143.7
"""online""",160.56
"""store""",187.01


### 15. **Per region**, the **earliest** `order_date` (min).

In [15]:
orders.group_by("region").agg(pl.col("order_date").min().alias("first_order"))

region,first_order
str,str
"""Middle East & Africa""","""2023-01-17"""
"""Asia Pacific""","""2023-01-02"""
"""North America""","""2023-01-01"""
"""Europe""","""2023-01-04"""
"""Latin America""","""2023-01-04"""


### 16. Count the number of **returns** (quantity < 0) per category.

In [16]:
orders.group_by("category").agg(
    n_returns=(pl.col("quantity") < 0).sum()
)

category,n_returns
str,u32
"""shirts""",7
"""mugs""",3
"""posters""",6
"""notebooks""",12
"""books""",7
"""pens""",6
"""stickers""",2


### 17. The **5 customers** with the highest total revenue.

In [17]:
orders.group_by("customer_id").agg(pl.col("revenue").sum()).sort("revenue", descending=True).head(5)

customer_id,revenue
str,f64
"""CUST-0064""",2309.26
"""CUST-0123""",2307.2
"""CUST-0079""",2148.88
"""CUST-0107""",1991.28
"""CUST-0075""",1984.65


### 18. **Window function:** add a column `category_total` next to every row, holding the total revenue of that row's category — *without* collapsing the table (use `.over("category")`).

In [18]:
orders.with_columns(
    category_total=pl.col("revenue").sum().over("category")
).select("order_id", "category", "revenue", "category_total").head()

order_id,category,revenue,category_total
str,str,f64,f64
"""ORD-000001""","""pens""",40.5,5819.12
"""ORD-000002""","""shirts""",89.4,54219.54
"""ORD-000003""","""books""",126.24,45408.32
"""ORD-000004""","""shirts""",631.22,54219.54
"""ORD-000005""","""posters""",79.86,28442.0


### 19. Using the window result, add `pct_of_category` = each order's revenue as a **percent of its category total**, rounded to 1 decimal.

In [19]:
orders.with_columns(
    pct_of_category=(
        pl.col("revenue") / pl.col("revenue").sum().over("category") * 100
    ).round(1)
).select("order_id", "category", "revenue", "pct_of_category").head()

order_id,category,revenue,pct_of_category
str,str,f64,f64
"""ORD-000001""","""pens""",40.5,0.7
"""ORD-000002""","""shirts""",89.4,0.2
"""ORD-000003""","""books""",126.24,0.3
"""ORD-000004""","""shirts""",631.22,1.2
"""ORD-000005""","""posters""",79.86,0.3
